# Step 11: Construction of the author-level survival dataset

**Paper:** Gendered dropout and scientific entry environments (HSR special issue *Computational Social Inequality Research in Practice*)

**Input:** `data/derived/filtered_data_recovered.csv` (one row per author × paper; output of `step10_merge_gender.ipynb`, step 10)
Columns: `AID, PMID, gender, begin_year, topics_num, PubYear`.

**Output (written to `OUT_DIR` = `data/derived/`):**

| file | content |
|---|---|
| `cox_data.csv` | one row per author (analysis population), raw entry variables + outcome for gap 2 and 3 — **input for `step12_cox_models.ipynb` (step 12)** |
| `topic_year.csv` | topic × year table (papers, authors by gender, maturity) — kept in `dataprep.duckdb`; the CSV export in section 3 is optional (commented out) |
| `sample_flow.csv` | number of authors after each selection step (for the methods section) |
| `diag_classification_by_entry_year_span.csv` | share of authors with F/M classification by entry year × career span (data-quality diagnostic) |
| `diag_gender_codes_by_entry_year.csv` | composition of gender codes (F/M/UNI/INI/UNK) by entry year |

**Design decisions:**

1. `begin_year` / `last_pubyear` are computed from **all** papers of an author (with or without topic).
2. The entry topic is chosen deterministically (most papers in the entry year; ties broken by a fixed hash of AID × topic), so the result is reproducible under parallel execution.
3. `first_year_papers` counts **distinct** PMIDs.
4. Topic maturity is computed from topic × year paper counts.
5. Additional entry variables for robustness checks: leave-one-out female share, female share among entrants, gender-classification rate in the entry topic-year, paper-weighted female share over all entry-year topics, field of the entry topic.
6. No transformations (centering, logs, dummies) here — they depend on the analysis sample and are done in `step12_cox_models.ipynb` (step 12).
7. A sample-flow table and data-quality diagnostics are written out (authors without any gender code count as unclassified).

## 0. Configuration

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import duckdb

BASE_DIR = os.path.abspath('..')                  # root of the replication package (notebooks run from code/)
INPUT_CSV = f'{BASE_DIR}/data/derived/filtered_data_recovered.csv'   # output of step 10
GENDER_BY_AID_CSV = f'{BASE_DIR}/data/derived/gender_by_aid.csv'   # step 10: source of the gender per AID -> `gender_source`
TOPIC_MAP_CSV = f'{BASE_DIR}/data/raw/openalex_topic_field_subfield_table.csv'   # OpenAlex topic -> subfield/field/domain
OUT_DIR = f'{BASE_DIR}/data/derived'               # output folder
DB_PATH = f'{OUT_DIR}/dataprep.duckdb'

DATA_END = 2023            # last publication year in the data
ENTRY_MIN, ENTRY_MAX = 1950, 2018   # entry cohorts kept (ENTRY_MAX leaves >= 5 years of follow-up)
GAPS = [2, 3]              # dropout = no publication in the GAP years after the last publication
MIN_TOPIC_SIZE = 20        # drop entry topic-years with <= 20 papers
MAX_FIRST_YEAR_PAPERS = 20 # drop authors with >= 20 papers in their first year (data artefacts)

os.makedirs(OUT_DIR, exist_ok=True)
con = duckdb.connect(DB_PATH)
con.execute("SET preserve_insertion_order = false")   # lowers memory use for large tables
# con.execute("SET memory_limit = '12GB'")             # uncomment and adapt if DuckDB runs out of memory

t0 = time.time()
def log(msg):
    print(f'[{time.time() - t0:7.0f}s] {msg}')

## 1. Load paper × author rows

* `gender` keeps the Wiki-Gendersort codes F, M, UNI, INI; `UNK` and empty strings become NULL.
* `topics_num` is stored as e.g. `10009.0` or `UNK` → integer topic id or NULL.
* Rows with `AID = 0` (no disambiguated author) are dropped.

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE raw AS
SELECT
    CAST(AID AS BIGINT)                                  AS aid,
    CAST(PMID AS BIGINT)                                 AS pmid,
    CASE WHEN gender IN ('F', 'M', 'UNI', 'INI') THEN gender END AS gender,
    CAST(TRY_CAST(topics_num AS DOUBLE) AS INTEGER)      AS topic,
    CAST(PubYear AS INTEGER)                             AS pubyear
FROM read_csv('{INPUT_CSV}', header = true, all_varchar = true)
WHERE AID IS NOT NULL AND AID <> '0'
""")
log(con.execute("SELECT count(*) AS n_rows, count(DISTINCT aid) AS n_authors, count(DISTINCT pmid) AS n_papers FROM raw").df().to_string(index=False))

## 2. Author table (all authors, any gender code)

`begin_year` = first publication year, `last_pubyear` = last publication year, both over **all** papers of the author.
Gender is assigned per AID upstream; the check below verifies that each AID carries exactly one gender code.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE authors AS
SELECT
    aid,
    min(pubyear)                    AS begin_year,
    max(pubyear)                    AS last_pubyear,
    min(gender)                     AS gender,          -- constant within AID (checked below)
    count(DISTINCT gender)          AS n_gender_codes,
    count(*) FILTER (WHERE gender IS NULL) AS n_rows_gender_null,
    count(DISTINCT pmid)            AS n_papers
FROM raw
GROUP BY aid
""")
check = con.execute("""
SELECT
    count(*) FILTER (WHERE n_gender_codes > 1)                          AS aids_with_conflicting_codes,
    count(*) FILTER (WHERE gender IS NOT NULL AND n_rows_gender_null > 0) AS aids_with_partly_missing_gender
FROM authors
""").df()
print(check.to_string(index=False))
assert check.iloc[0, 0] == 0, 'gender is not constant within AID -> revisit gender assignment'

## 3. Topic × year table

For every topic (OpenAlex topic id) and year:

* `n_papers`: distinct papers (topic size),
* `n_authors_all`, `n_f`, `n_m`: distinct active authors (all codes / female / male),
* `n_f_entrants`, `n_m_entrants`: active authors whose first publication year is this year,
* `median_pubyear`, `maturity`: median publication year of all papers in the topic up to and including this year and `year - median_pubyear`
  (note that the data start in 1950, so maturity is left-truncated for early years).

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE topic_year_papers AS
SELECT topic, pubyear AS year, count(DISTINCT pmid) AS n_papers
FROM raw
WHERE topic IS NOT NULL
GROUP BY topic, pubyear
""")

con.execute("""
CREATE OR REPLACE TABLE topic_year_authors AS
WITH ta AS (
    SELECT DISTINCT r.topic, r.pubyear AS year, r.aid, a.gender, (a.begin_year = r.pubyear) AS is_entrant
    FROM raw r
    JOIN authors a USING (aid)
    WHERE r.topic IS NOT NULL
)
SELECT
    topic, year,
    count(*)                                              AS n_authors_all,
    count(*) FILTER (WHERE gender = 'F')                  AS n_f,
    count(*) FILTER (WHERE gender = 'M')                  AS n_m,
    count(*) FILTER (WHERE gender = 'F' AND is_entrant)   AS n_f_entrants,
    count(*) FILTER (WHERE gender = 'M' AND is_entrant)   AS n_m_entrants
FROM ta
GROUP BY topic, year
""")

# maturity: median of the publication-year distribution of all papers in the topic up to `year`
# (average of the two middle values for an even number of papers = DuckDB median / quantile_cont)
con.execute("""
CREATE OR REPLACE TABLE topic_year_maturity AS
WITH pairs AS (
    SELECT t.topic, t.year AS ref_year, p.year AS pub_year, p.n_papers
    FROM topic_year_papers t
    JOIN topic_year_papers p ON p.topic = t.topic AND p.year <= t.year
),
cum AS (
    SELECT *,
        sum(n_papers) OVER (PARTITION BY topic, ref_year ORDER BY pub_year) AS cum_n,
        sum(n_papers) OVER (PARTITION BY topic, ref_year)                   AS total_n
    FROM pairs
)
SELECT
    topic,
    ref_year AS year,
    (min(pub_year) FILTER (WHERE cum_n > floor((total_n - 1) / 2))
   + min(pub_year) FILTER (WHERE cum_n > floor(total_n / 2))) / 2.0       AS median_pubyear
FROM cum
GROUP BY topic, ref_year
""")

con.execute("""
CREATE OR REPLACE TABLE topic_year AS
SELECT
    p.topic, p.year, p.n_papers,
    a.n_authors_all, a.n_f, a.n_m, a.n_f_entrants, a.n_m_entrants,
    m.median_pubyear,
    p.year - m.median_pubyear                      AS maturity,
    a.n_f / nullif(a.n_f + a.n_m, 0)               AS female_share,
    (a.n_f + a.n_m) / nullif(a.n_authors_all, 0)   AS class_rate
FROM topic_year_papers p
JOIN topic_year_authors a USING (topic, year)
JOIN topic_year_maturity m USING (topic, year)
""")
#con.execute(f"COPY topic_year TO '{OUT_DIR}/topic_year.csv' (HEADER, DELIMITER ',')")
log(con.execute("SELECT count(*) AS topic_years, count(DISTINCT topic) AS topics FROM topic_year").df().to_string(index=False))

## 4. Entry topic and entry-year productivity

* Entry year = `begin_year`. Entry topic = topic with most distinct papers of the author in the entry year.
* Ties (≈10 % of authors) are broken by `hash(aid, topic)`: deterministic and reproducible, but unrelated to topic characteristics.
* `first_year_papers` = distinct papers in the entry year (all papers, with or without topic).
* `female_share_entry_w`: paper-weighted mean female share over **all** topics of the entry year (robustness check for authors with several entry topics).

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE entry_topic_counts AS
SELECT r.aid, r.topic, count(DISTINCT r.pmid) AS n_entry_papers_topic
FROM raw r
JOIN authors a ON a.aid = r.aid AND r.pubyear = a.begin_year
WHERE r.topic IS NOT NULL
GROUP BY r.aid, r.topic
""")

con.execute("""
CREATE OR REPLACE TABLE entry_topic AS
SELECT
    aid,
    topic                                         AS entry_topic,
    n_entry_papers_topic,
    count(*) OVER (PARTITION BY aid)              AS n_entry_topics,
    count(*) FILTER (WHERE n_entry_papers_topic = max_n) OVER (PARTITION BY aid) AS n_tied_top_topics
FROM (
    SELECT *, max(n_entry_papers_topic) OVER (PARTITION BY aid) AS max_n
    FROM entry_topic_counts
)
QUALIFY row_number() OVER (PARTITION BY aid ORDER BY n_entry_papers_topic DESC, hash(aid, topic)) = 1
""")

con.execute("""
CREATE OR REPLACE TABLE entry_weighted AS
SELECT
    e.aid,
    sum(e.n_entry_papers_topic * ty.female_share) / sum(e.n_entry_papers_topic) AS female_share_entry_w
FROM entry_topic_counts e
JOIN authors a USING (aid)
JOIN topic_year ty ON ty.topic = e.topic AND ty.year = a.begin_year
GROUP BY e.aid
""")

con.execute("""
CREATE OR REPLACE TABLE first_year AS
SELECT r.aid, count(DISTINCT r.pmid) AS first_year_papers
FROM raw r
JOIN authors a ON a.aid = r.aid AND r.pubyear = a.begin_year
GROUP BY r.aid
""")
print(con.execute("""
SELECT
    avg((n_entry_topics > 1)::INT)    AS share_multiple_entry_topics,
    avg((n_tied_top_topics > 1)::INT) AS share_tied_entry_topic
FROM entry_topic
""").df().to_string(index=False))

## 4b. Source of the gender classification

`gender_source` = `pkg` if F/M stems from the author's PKG forename (step 7), `recovered` if F/M stems
from the forename recovered from MEDLINE (steps 8–9); taken from `gender_by_aid.csv` (step 10). The two sources differ strongly in the share of women they
produce (≈ 47 % vs. ≈ 27 % among 2002–2018 entrants) and in the gender gap they imply, so the source is kept for
robustness checks (spec `check_pkg_2002` in step 12).

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE pkg_gender AS
SELECT CAST(AID AS BIGINT) AS aid,
       max(CASE WHEN gender_source = 'pkg' THEN 1 ELSE 0 END) AS pkg_classified
FROM read_csv('{GENDER_BY_AID_CSV}', header = true, all_varchar = true)
WHERE AID IS NOT NULL AND AID <> '0'
GROUP BY 1
""")

## 5. Author-level dataset with entry variables and outcome

**Entry variables** (all measured in the entry topic in the entry year):

| variable | definition |
|---|---|
| `topic_size_entry` | distinct papers in entry topic-year |
| `topic_maturity_entry` | entry year − median publication year of all topic papers up to the entry year |
| `female_share_entry` | female / (female + male) active authors in entry topic-year (**main measure**; includes the author her/himself) |
| `female_share_entry_loo` | same, leaving the focal author out |
| `female_share_entrants_loo` | female share among the *other* entrants of the entry topic-year |
| `female_share_entry_w` | paper-weighted female share over all entry-year topics |
| `class_rate_entry` | share of active authors in the entry topic-year with F/M classification |
| `first_year_papers` | distinct papers in the entry year |
| `gender_source` | `pkg` (F/M from the PKG forename) or `recovered` (F/M from the forename recovered from MEDLINE) |

**Outcome** for each gap *g* ∈ `GAPS`: dropout is observed if `last_pubyear + g <= DATA_END` (g years without publication are observed).
`duration_g` = `last_pubyear + g − begin_year` for events, `DATA_END − begin_year` for censored authors.

In [ ]:
outcome_sql = ',\n'.join(
    f"""    ((a.last_pubyear + {g}) <= {DATA_END})::INT AS event_{g}y,
    CASE WHEN (a.last_pubyear + {g}) <= {DATA_END} THEN a.last_pubyear + {g} - a.begin_year
         ELSE {DATA_END} - a.begin_year END AS duration_{g}y"""
    for g in GAPS)

con.execute(f"""
CREATE OR REPLACE TABLE author_all AS
SELECT
    a.aid AS AID,
    a.gender,
    (a.gender = 'F')::INT                                   AS gender_f,
    a.begin_year,
    a.last_pubyear,
    a.n_papers,
    (a.begin_year = a.last_pubyear)::INT                    AS one_year,
    e.entry_topic                                           AS topics_num,
    e.n_entry_topics,
    (e.n_tied_top_topics > 1)::INT                          AS entry_topic_tied,
    f.first_year_papers,
    ty.n_papers                                             AS topic_size_entry,
    ty.maturity                                             AS topic_maturity_entry,
    ty.female_share                                         AS female_share_entry,
    (ty.n_f - (a.gender = 'F')::INT) / nullif(ty.n_f + ty.n_m - 1, 0)                      AS female_share_entry_loo,
    (ty.n_f_entrants - (a.gender = 'F')::INT) / nullif(ty.n_f_entrants + ty.n_m_entrants - 1, 0) AS female_share_entrants_loo,
    w.female_share_entry_w,
    ty.class_rate                                           AS class_rate_entry,
    ty.n_f + ty.n_m                                         AS n_classified_entry,
    CASE WHEN a.gender NOT IN ('F', 'M') OR a.gender IS NULL THEN NULL
         WHEN pg.pkg_classified = 1 THEN 'pkg' ELSE 'recovered' END AS gender_source,
{outcome_sql}
FROM authors a
LEFT JOIN entry_topic e    ON e.aid = a.aid
LEFT JOIN first_year f     ON f.aid = a.aid
LEFT JOIN entry_weighted w ON w.aid = a.aid
LEFT JOIN topic_year ty    ON ty.topic = e.entry_topic AND ty.year = a.begin_year
LEFT JOIN pkg_gender pg    ON pg.aid = a.aid
""")
log('author_all built')

## 6. Sample selection (with flow table)

In [ ]:
steps = [
    ('all authors (AID != 0)',                               'TRUE'),
    ('gender classified as F or M',                          "gender IN ('F', 'M')"),
    ('entry year has at least one paper with topic',         'topics_num IS NOT NULL'),
    (f'entry year {ENTRY_MIN}-{ENTRY_MAX}',                   f'begin_year BETWEEN {ENTRY_MIN} AND {ENTRY_MAX}'),
    (f'entry topic-year > {MIN_TOPIC_SIZE} papers',           f'topic_size_entry > {MIN_TOPIC_SIZE}'),
    (f'< {MAX_FIRST_YEAR_PAPERS} papers in first year',       f'first_year_papers < {MAX_FIRST_YEAR_PAPERS}'),
]
flow, conds = [], []
for label, cond in steps:
    conds.append(cond)
    where = ' AND '.join(conds)
    n = con.execute(f"""SELECT count(*) AS n, sum(gender_f) AS n_f, sum(one_year) AS n_one_year
                        FROM author_all WHERE {where}""").df().iloc[0]
    flow.append({'step': label, 'n_authors': int(n['n']),
                 'n_female': int(n['n_f']) if pd.notna(n['n_f']) else None,
                 'n_one_year_authors': int(n['n_one_year'])})
flow = pd.DataFrame(flow)
flow['n_authors_publishing_2plus_years'] = flow['n_authors'] - flow['n_one_year_authors']
flow.to_csv(f'{OUT_DIR}/sample_flow.csv', index=False)
print(flow.to_string(index=False))
FINAL_WHERE = ' AND '.join(conds)

## 7. Field of the entry topic (for stratified robustness models) and export

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE topic_map AS
SELECT CAST(topic_id AS INTEGER) AS topic_id,
       any_value(subfield_name) AS subfield_name,
       any_value(field_name)    AS field_name,
       any_value(domain_name)   AS domain_name
FROM read_csv('{TOPIC_MAP_CSV}', header = true)
GROUP BY topic_id
""")

con.execute(f"""
COPY (
    SELECT s.*, m.field_name AS field, m.subfield_name AS subfield, m.domain_name AS domain
    FROM author_all s
    LEFT JOIN topic_map m ON m.topic_id = s.topics_num
    WHERE {FINAL_WHERE}
    ORDER BY AID
) TO '{OUT_DIR}/cox_data.csv' (HEADER, DELIMITER ',')
""")
log('cox_data.csv written')

## 8. Data-quality diagnostics: is gender classification related to career length?

If the probability that an author receives an F/M classification depends on how long the author stays active,
the classified population is selected on the outcome. The table reports, by entry year and career span
(`last_pubyear − begin_year`), the share of authors with F/M classification and the female share among them.
A strong jump around 2002 (NLM starts to record full first names) indicates outcome-dependent classification
for earlier cohorts.

In [ ]:
diag = con.execute(f"""
SELECT
    begin_year,
    CASE WHEN last_pubyear - begin_year = 0 THEN '0'
         WHEN last_pubyear - begin_year <= 4 THEN '1-4'
         WHEN last_pubyear - begin_year <= 9 THEN '5-9'
         ELSE '10+' END                                      AS career_span,
    count(*)                                                 AS n_authors,
    avg(coalesce(gender IN ('F', 'M'), false)::INT)          AS share_classified_fm,  -- NULL gender counts as unclassified
    sum((gender = 'F')::INT) / nullif(sum((gender IN ('F', 'M'))::INT), 0) AS female_share_among_classified
FROM authors
WHERE begin_year <= {ENTRY_MAX}
GROUP BY ALL
ORDER BY begin_year, career_span
""").df()
diag.to_csv(f'{OUT_DIR}/diag_classification_by_entry_year_span.csv', index=False)
print(diag[diag.begin_year.isin([1990, 1995, 1999, 2001, 2002, 2005, 2010])].pivot(
    index='begin_year', columns='career_span', values=['share_classified_fm', 'female_share_among_classified']).round(3))

codes = con.execute(f"""
SELECT begin_year, coalesce(gender, 'UNK') AS gender_code, count(*) AS n_authors
FROM authors WHERE begin_year <= {ENTRY_MAX}
GROUP BY ALL ORDER BY ALL
""").df()
codes.to_csv(f'{OUT_DIR}/diag_gender_codes_by_entry_year.csv', index=False)

## 9. Quick checks of the exported dataset

In [ ]:
cox = pd.read_csv(f'{OUT_DIR}/cox_data.csv')
print(cox.shape)
print(cox[['begin_year', 'first_year_papers', 'female_share_entry', 'topic_size_entry',
           'topic_maturity_entry', 'class_rate_entry', 'event_2y', 'duration_2y']].describe().T.round(3).to_string())
print('\nCorrelations (analysis sample):')
print(cox[['begin_year', 'female_share_entry', 'topic_size_entry', 'topic_maturity_entry', 'gender_f']].corr().round(2).to_string())

In [ ]:
con.close()
log('done')